In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/lyk1zm/llm-classification-finetuning3/sample_submission.csv
/kaggle/input/datasets/lyk1zm/llm-classification-finetuning3/train.csv
/kaggle/input/datasets/lyk1zm/llm-classification-finetuning3/test.csv
/kaggle/input/notebooks/lyk1zm/04-transformer-submission/preference_data.py
/kaggle/input/notebooks/lyk1zm/04-transformer-submission/__results__.html
/kaggle/input/notebooks/lyk1zm/04-transformer-submission/submission.csv
/kaggle/input/notebooks/lyk1zm/04-transformer-submission/__notebook__.ipynb
/kaggle/input/notebooks/lyk1zm/04-transformer-submission/__output__.json
/kaggle/input/notebooks/lyk1zm/04-transformer-submission/custom.css
/kaggle/input/notebooks/lyk1zm/04-transformer-submission/__pycache__/preference_data.cpython-312.pyc
/kaggle/input/notebooks/lyk1zm/01-data-and-validation/__results__.html
/kaggle/input/notebooks/lyk1zm/01-data-and-validation/__notebook__.ipynb
/kaggle/input/notebooks/lyk1zm/01-data-and-validation/__output__.json
/kaggle/input/notebook

In [2]:
import os

os.environ["TOKENIZERS_PARALLELISM"] = "false"

import gc
import json
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import transformers

from IPython.display import display
from sklearn.metrics import log_loss, accuracy_score
from sklearn.model_selection import StratifiedGroupKFold
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    get_linear_schedule_with_warmup,
)

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("CUDA available:", torch.cuda.is_available())

assert torch.cuda.is_available(), (
    "GPU не обнаружена. Выбери P100 и перезапусти сессию."
)

print("GPU:", torch.cuda.get_device_name(0))
print(
    "GPU memory GB:",
    round(
        torch.cuda.get_device_properties(0).total_memory / 1024**3,
        2,
    ),
)

device = torch.device("cuda")

PyTorch: 2.10.0+cu128
Transformers: 5.0.0
CUDA available: True
GPU: Tesla T4
GPU memory GB: 14.56


In [3]:
SEED = 42

MODEL_NAME = "distilbert/distilbert-base-multilingual-cased"

PROMPT_TOKENS = 96
ANSWER_TOKENS = 142

TRAIN_BATCH_SIZE = 2
EVAL_BATCH_SIZE = 16
GRAD_ACCUM_STEPS = 8

EPOCHS = 2
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01

# True — короткая проверка всей цепочки.
# False — полноценный эксперимент.
SMOKE_TEST = False

TARGET_COLUMNS = [
    "winner_model_a",
    "winner_model_b",
    "winner_tie",
]

WORK_DIR = Path("/kaggle/working")
MODEL_DIR = WORK_DIR / "preference_encoder"
MODEL_DIR.mkdir(parents=True, exist_ok=True)


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


seed_everything(SEED)

print(
    "Effective batch size:",
    TRAIN_BATCH_SIZE * GRAD_ACCUM_STEPS,
)

Effective batch size: 16


In [4]:
%%writefile /kaggle/working/preference_data.py

import json
import random

import numpy as np
import pandas as pd

from torch.utils.data import Dataset


TEXT_COLUMNS = ["prompt", "response_a", "response_b"]


def parse_turns(value):
    turns = json.loads(value)

    if not isinstance(turns, list):
        raise ValueError("Expected a JSON list")

    if any(
        turn is not None and not isinstance(turn, str)
        for turn in turns
    ):
        raise ValueError("Unexpected turn type")

    return ["" if turn is None else turn for turn in turns]


def prepare_frame(df):
    result = df.copy()

    for column in TEXT_COLUMNS:
        turns = result[column].map(parse_turns)

        # Маркеры сохраняют хотя бы явные границы реплик.
        result[f"{column}_text"] = turns.map(
            lambda items: "\n\n".join(
                f"Turn {i + 1}: {text}"
                for i, text in enumerate(items)
            )
        )

        if column == "prompt":
            # Совпадает с логикой групп baseline.
            result["prompt_group"] = turns.map(
                lambda items: json.dumps(
                    [" ".join(text.split()) for text in items],
                    ensure_ascii=False,
                )
            )

    return result


def encode_frame(
    df,
    tokenizer,
    prompt_tokens,
    answer_tokens,
    batch_size=256,
):
    encoded = {}
    limits = {
        "prompt": prompt_tokens,
        "response_a": answer_tokens,
        "response_b": answer_tokens,
    }

    for column in TEXT_COLUMNS:
        texts = df[f"{column}_text"].tolist()
        sequences = []

        for left in range(0, len(texts), batch_size):
            batch = tokenizer(
                texts[left:left + batch_size],
                add_special_tokens=False,
                truncation=True,
                max_length=limits[column],
                padding=False,
                return_attention_mask=False,
                return_token_type_ids=False,
            )

            sequences.extend(batch["input_ids"])

        encoded[column] = sequences

    return encoded


class PreferenceDataset(Dataset):
    def __init__(
        self,
        encoded,
        cls_token_id,
        sep_token_id,
        labels=None,
        random_swap=False,
        fixed_swap=False,
    ):
        self.encoded = encoded
        self.cls_token_id = cls_token_id
        self.sep_token_id = sep_token_id
        self.labels = (
            None
            if labels is None
            else np.asarray(labels, dtype=np.int64)
        )
        self.random_swap = random_swap
        self.fixed_swap = fixed_swap

    def __len__(self):
        return len(self.encoded["prompt"])

    def __getitem__(self, index):
        prompt = self.encoded["prompt"][index]
        answer_a = self.encoded["response_a"][index]
        answer_b = self.encoded["response_b"][index]

        swap = self.fixed_swap or (
            self.random_swap and random.random() < 0.5
        )

        if swap:
            answer_a, answer_b = answer_b, answer_a

        input_ids = (
            [self.cls_token_id]
            + prompt
            + [self.sep_token_id]
            + answer_a
            + [self.sep_token_id]
            + answer_b
            + [self.sep_token_id]
        )

        item = {
            "input_ids": input_ids,
            "attention_mask": [1] * len(input_ids),
        }

        if self.labels is not None:
            label = int(self.labels[index])

            if swap:
                label = [1, 0, 2][label]

            item["labels"] = label

        return item

Writing /kaggle/working/preference_data.py


In [5]:
from preference_data import (
    prepare_frame,
    encode_frame,
    PreferenceDataset,
)

possible_dirs = [
    Path("/kaggle/input/competitions/llm-classification-finetuning"),
    Path("/kaggle/input/llm-classification-finetuning"),
]

DATA_DIR = next(
    (p for p in possible_dirs if (p / "train.csv").exists()),
    None,
)

assert DATA_DIR is not None, "Подключи данные соревнования."

train = pd.read_csv(DATA_DIR / "train.csv")

assert train[TARGET_COLUMNS].isin([0, 1]).all().all()
assert train[TARGET_COLUMNS].sum(axis=1).eq(1).all()
assert train["id"].is_unique

train["label"] = (
    train[TARGET_COLUMNS].to_numpy().argmax(axis=1)
)

train = prepare_frame(train)

splitter = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=SEED,
)

train_idx, valid_idx = next(
    splitter.split(
        train,
        train["label"],
        groups=train["prompt_group"],
    )
)

train_part = train.iloc[train_idx].copy()
valid_part = train.iloc[valid_idx].copy()

assert set(train_part["prompt_group"]).isdisjoint(
    set(valid_part["prompt_group"])
)

split_info = train[["id"]].copy()
split_info["split"] = "train"
split_info.loc[valid_part.index, "split"] = "valid"

split_info.to_csv(WORK_DIR / "split.csv", index=False)

if SMOKE_TEST:
    train_part = train_part.sample(
        n=min(512, len(train_part)),
        random_state=SEED,
    )
    valid_part = valid_part.sample(
        n=min(256, len(valid_part)),
        random_state=SEED,
    )

y_train = train_part["label"].to_numpy()
y_valid = valid_part["label"].to_numpy()

print("Train:", len(train_part))
print("Validation:", len(valid_part))

if SMOKE_TEST:
    print("SMOKE TEST: эти метрики не сравниваем с полным baseline.")

Train: 45599
Validation: 11878


In [6]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    id2label={
        0: "A",
        1: "B",
        2: "Tie",
    },
    label2id={
        "A": 0,
        "B": 1,
        "Tie": 2,
    },
)

assert tokenizer.cls_token_id is not None
assert tokenizer.sep_token_id is not None

model.to(device)

parameter_count = sum(
    parameter.numel()
    for parameter in model.parameters()
)

print("Parameters:", f"{parameter_count:,}")
print("Model device:", next(model.parameters()).device)

config.json:   0%|          | 0.00/466 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/542M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-multilingual-cased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Parameters: 135,326,979
Model device: cuda:0


In [7]:
start = time.perf_counter()

train_encoded = encode_frame(
    train_part,
    tokenizer,
    prompt_tokens=PROMPT_TOKENS,
    answer_tokens=ANSWER_TOKENS,
)

valid_encoded = encode_frame(
    valid_part,
    tokenizer,
    prompt_tokens=PROMPT_TOKENS,
    answer_tokens=ANSWER_TOKENS,
)

print(
    "Tokenization seconds:",
    round(time.perf_counter() - start, 1),
)

dataset_kwargs = {
    "cls_token_id": tokenizer.cls_token_id,
    "sep_token_id": tokenizer.sep_token_id,
}

train_dataset = PreferenceDataset(
    train_encoded,
    labels=y_train,
    random_swap=True,
    **dataset_kwargs,
)

valid_dataset = PreferenceDataset(
    valid_encoded,
    **dataset_kwargs,
)

valid_swapped_dataset = PreferenceDataset(
    valid_encoded,
    fixed_swap=True,
    **dataset_kwargs,
)

collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    pad_to_multiple_of=8,
    return_tensors="pt",
)

train_loader = DataLoader(
    train_dataset,
    batch_size=TRAIN_BATCH_SIZE,
    shuffle=True,
    collate_fn=collator,
    num_workers=0,
    pin_memory=True,
)

valid_loader = DataLoader(
    valid_dataset,
    batch_size=EVAL_BATCH_SIZE,
    shuffle=False,
    collate_fn=collator,
    num_workers=0,
    pin_memory=True,
)

valid_swapped_loader = DataLoader(
    valid_swapped_dataset,
    batch_size=EVAL_BATCH_SIZE,
    shuffle=False,
    collate_fn=collator,
    num_workers=0,
    pin_memory=True,
)

example_batch = next(iter(train_loader))

print({
    key: tuple(value.shape)
    for key, value in example_batch.items()
})

assert example_batch["input_ids"].shape[1] <= 384

del example_batch

Tokenization seconds: 97.5
{'input_ids': (2, 344), 'attention_mask': (2, 344), 'labels': (2,)}


In [8]:
@torch.inference_mode()
def predict_probabilities(model, loader, description="Predict"):
    model.eval()
    predictions = []

    for batch in tqdm(loader, desc=description):
        batch = {
            key: value.to(device, non_blocking=True)
            for key, value in batch.items()
        }

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
        ):
            logits = model(**batch).logits

        probabilities = (
            torch.softmax(logits.float(), dim=-1)
            .cpu()
            .numpy()
        )

        predictions.append(probabilities)

    return np.vstack(predictions)

In [9]:
import math

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

updates_per_epoch = math.ceil(
    len(train_loader) / GRAD_ACCUM_STEPS
)

total_updates = updates_per_epoch * EPOCHS

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.1 * total_updates),
    num_training_steps=total_updates,
)

scaler = torch.amp.GradScaler("cuda")

history = []
best_score = float("inf")

training_config = {
    "base_model": MODEL_NAME,
    "seed": SEED,
    "prompt_tokens": PROMPT_TOKENS,
    "answer_tokens": ANSWER_TOKENS,
    "max_length": PROMPT_TOKENS + 2 * ANSWER_TOKENS + 4,
    "train_batch_size": TRAIN_BATCH_SIZE,
    "gradient_accumulation_steps": GRAD_ACCUM_STEPS,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "epochs_planned": EPOCHS,
    "random_swap_training": True,
    "smoke_test": SMOKE_TEST,
    "torch_version": torch.__version__,
    "transformers_version": transformers.__version__,
}

for epoch in range(EPOCHS):
    epoch_start = time.perf_counter()

    model.train()
    optimizer.zero_grad(set_to_none=True)

    loss_sum = 0.0
    example_count = 0

    progress = tqdm(
        enumerate(train_loader),
        total=len(train_loader),
        desc=f"Epoch {epoch + 1}/{EPOCHS}",
    )

    for step, batch in progress:
        batch = {
            key: value.to(device, non_blocking=True)
            for key, value in batch.items()
        }

        # Размер текущей группы накопления градиентов.
        # Последняя группа может быть короче остальных.
        group_start = (
            step // GRAD_ACCUM_STEPS
        ) * GRAD_ACCUM_STEPS

        group_end = min(
            group_start + GRAD_ACCUM_STEPS,
            len(train_loader),
        )

        group_examples = (
            min(
                group_end * TRAIN_BATCH_SIZE,
                len(train_dataset),
            )
            - group_start * TRAIN_BATCH_SIZE
        )

        batch_size = batch["labels"].shape[0]

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
        ):
            outputs = model(**batch)
            raw_loss = outputs.loss

            # Корректное взвешивание, в том числе последнего батча.
            loss = raw_loss * batch_size / group_examples

        if not torch.isfinite(raw_loss):
            raise RuntimeError(
                "Получен NaN/Inf loss. Останови запуск и пришли ошибку."
            )

        scaler.scale(loss).backward()

        loss_sum += raw_loss.item() * batch_size
        example_count += batch_size

        should_update = (
            (step + 1) % GRAD_ACCUM_STEPS == 0
            or (step + 1) == len(train_loader)
        )

        if should_update:
            scaler.unscale_(optimizer)

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=1.0,
            )

            old_scale = scaler.get_scale()

            scaler.step(optimizer)
            scaler.update()

            # Если AMP пропустил обновление из-за overflow,
            # не двигаем scheduler.
            if scaler.get_scale() >= old_scale:
                scheduler.step()

            optimizer.zero_grad(set_to_none=True)

        progress.set_postfix(
            loss=f"{loss_sum / example_count:.4f}",
            lr=f"{scheduler.get_last_lr()[0]:.2e}",
        )

    original_probabilities = predict_probabilities(
        model,
        valid_loader,
        description="Validation",
    )

    swapped_probabilities = predict_probabilities(
        model,
        valid_swapped_loader,
        description="Validation swapped",
    )[:, [1, 0, 2]]

    tta_probabilities = (
        original_probabilities + swapped_probabilities
    ) / 2

    original_score = log_loss(
        y_valid,
        original_probabilities,
        labels=[0, 1, 2],
    )

    tta_score = log_loss(
        y_valid,
        tta_probabilities,
        labels=[0, 1, 2],
    )

    use_tta = tta_score < original_score

    selected_probabilities = (
        tta_probabilities
        if use_tta
        else original_probabilities
    )

    selected_score = min(original_score, tta_score)

    row = {
        "epoch": epoch + 1,
        "train_loss": loss_sum / example_count,
        "valid_log_loss": original_score,
        "valid_tta_log_loss": tta_score,
        "selected_accuracy": accuracy_score(
            y_valid,
            selected_probabilities.argmax(axis=1),
        ),
        "swap_gap": float(
            np.abs(
                original_probabilities - swapped_probabilities
            ).mean()
        ),
        "epoch_seconds": time.perf_counter() - epoch_start,
    }

    history.append(row)

    display(pd.DataFrame(history))

    pd.DataFrame(history).to_csv(
        WORK_DIR / "training_history.csv",
        index=False,
    )

    if selected_score < best_score:
        best_score = selected_score

        model.save_pretrained(MODEL_DIR)
        tokenizer.save_pretrained(MODEL_DIR)

        inference_config = {
            **training_config,
            "best_epoch": epoch + 1,
            "use_swap_tta": bool(use_tta),
            "validation_log_loss": float(best_score),
        }

        with open(
            MODEL_DIR / "inference_config.json",
            "w",
            encoding="utf-8",
        ) as file:
            json.dump(
                inference_config,
                file,
                ensure_ascii=False,
                indent=2,
            )

        validation_output = pd.DataFrame({
            "id": valid_part["id"].to_numpy(),
            "label": y_valid,
            "p_a": selected_probabilities[:, 0],
            "p_b": selected_probabilities[:, 1],
            "p_tie": selected_probabilities[:, 2],
        })

        validation_output.to_csv(
            WORK_DIR / "transformer_validation_predictions.csv",
            index=False,
        )

        print(
            f"Saved best checkpoint: "
            f"epoch={epoch + 1}, "
            f"log_loss={best_score:.5f}, "
            f"TTA={use_tta}"
        )

    gc.collect()
    torch.cuda.empty_cache()

Epoch 1/2:   0%|          | 0/22800 [00:00<?, ?it/s]

Validation:   0%|          | 0/743 [00:00<?, ?it/s]

Validation swapped:   0%|          | 0/743 [00:00<?, ?it/s]

,epoch,train_loss,valid_log_loss,valid_tta_log_loss,selected_accuracy,swap_gap,epoch_seconds
0,1,1.079116,1.061243,1.047965,0.448055,0.080081,854.684724


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Validation:   0%|          | 0/743 [00:00<?, ?it/s]

Validation swapped:   0%|          | 0/743 [00:00<?, ?it/s]

,epoch,train_loss,valid_log_loss,valid_tta_log_loss,selected_accuracy,swap_gap,epoch_seconds
0,1,1.079116,1.061243,1.047965,0.448055,0.080081,854.684724
1,2,1.039940,1.040262,1.035032,0.459926,0.054226,857.205686


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved best checkpoint: epoch=2, log_loss=1.03503, TTA=True


In [10]:
import shutil

shutil.copy2(
    WORK_DIR / "preference_data.py",
    MODEL_DIR / "preference_data.py",
)

with open(
    MODEL_DIR / "inference_config.json",
    encoding="utf-8",
) as file:
    saved_config = json.load(file)

comparison = pd.DataFrame([
    {
        "model": "class_priors",
        "validation_log_loss": 1.097638,
    },
    {
        "model": "tfidf_lr_C=0.3_swap_tta",
        "validation_log_loss": 1.034430,
    },
    {
        "model": "multilingual_distilbert",
        "validation_log_loss": saved_config["validation_log_loss"],
    },
])

if not SMOKE_TEST:
    display(
        comparison.sort_values("validation_log_loss")
    )

    comparison.to_csv(
        WORK_DIR / "model_comparison.csv",
        index=False,
    )
else:
    print("Smoke test завершён. Сравнение с baseline неприменимо.")

print("Best checkpoint:", MODEL_DIR)

for path in sorted(MODEL_DIR.iterdir()):
    print(
        path.name,
        round(path.stat().st_size / 1024**2, 2),
        "MB",
    )

,model,validation_log_loss
1,tfidf_lr_C=0.3_swap_tta,1.034430
2,multilingual_distilbert,1.035032
0,class_priors,1.097638


Best checkpoint: /kaggle/working/preference_encoder
config.json 0.0 MB
inference_config.json 0.0 MB
model.safetensors 516.24 MB
preference_data.py 0.0 MB
tokenizer.json 2.78 MB
tokenizer_config.json 0.0 MB
